
## MTGFlow notebook adapted for an MTAD-GAT / mtad-tools project folder

This notebook is designed to be **copied into the same folder** where your `mtad_gat_notebook.ipynb` lives, so you can:
- reuse the **same YAML configs** (`SMD.yaml`, `mtad_gat.yaml`, etc.),
- reuse the **same dataset loader utilities** from your mtad-tools code (`load_dataset`, `generate_windows`),
- train and evaluate **MTGFlow** on the same data, step-by-step.

### Folder expectation (recommended)
Place this notebook in your MTAD-GAT project directory that contains:
- `dataloader.py`
- `data_preprocess.py`
- your YAML config files (`SMD.yaml`, `mtad_gat.yaml`, etc.)

And also place the MTGFlow repo folder alongside it, e.g.:
- `./MTGFLOW-main/` (the authors’ GitHub code)

If your layout differs, update `MTGFLOW_REPO_PATH` below.

---


In [13]:

# %%
# 0) Imports and environment setup

import os
import sys
import math
import random
import logging
import warnings
import yaml
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


Using device: cpu



## 1) Paths and config selection

These mirror the way your MTAD-GAT notebook loads YAML configs.

- `EXP_ID` is typically one of the experiment keys in `mtad_gat.yaml` (e.g., `mtad_gat_SMD`).
- `DATASET_CFG_FILE` is your dataset YAML (e.g., `SMD.yaml`).

If you want to run a different dataset (WADI, SWaT, etc.), use the corresponding YAML you already have.


In [14]:

# %%
# 1) Config selection (edit these)

EXP_ID = "mtad_gat_SMD"
DATASET_CFG_FILE = "SMD.yaml"

# Path to MTGFlow repo (expected to be next to this notebook)
MTGFLOW_REPO_PATH = "./MTGFLOW-main"   # <-- change if needed

# For SMD, mtad-tools often loads a list of entities (machines).
# MTGFlow is multivariate; we typically train per-entity (machine) treating features as sensors.
ENTITY_INDEX = 0  # choose which entity to run, e.g. 0 for machine-1, 1 for machine-2, etc.

# Training hyperparameters for MTGFlow (you can change freely)
MTGFLOW_EPOCHS = 40
MTGFLOW_LR = 2e-3
MTGFLOW_WEIGHT_DECAY = 5e-4
MTGFLOW_BATCH_SIZE = 256
GRAD_CLIP = 1.0

print("EXP_ID:", EXP_ID)
print("DATASET_CFG_FILE:", DATASET_CFG_FILE)
print("MTGFLOW_REPO_PATH:", MTGFLOW_REPO_PATH)


EXP_ID: mtad_gat_SMD
DATASET_CFG_FILE: SMD.yaml
MTGFLOW_REPO_PATH: ./MTGFLOW-main



## 2) Load YAML configs (same pattern as MTAD-GAT)

This cell reads:
- model config from `mtad_gat.yaml` (to reuse `window_size`, `stride`, etc.)
- dataset config from `SMD.yaml` (to reuse `data_root`, `entities`, etc.)


In [15]:

# %%
def load_model_config(model_cfg_path="mtad_gat.yaml", exp_id="mtad_gat_SMD"):
    with open(model_cfg_path, "r") as f:
        cfg_all = yaml.safe_load(f)

    if "Base" in cfg_all:
        base = cfg_all["Base"] or {}
    else:
        base = {}

    if exp_id not in cfg_all:
        raise ValueError(f"Experiment id '{exp_id}' not found. Available: {list(cfg_all.keys())}")

    merged = dict(base)
    merged.update(cfg_all[exp_id] or {})
    return merged

def load_dataset_config(dataset_cfg_path="SMD.yaml"):
    with open(dataset_cfg_path, "r") as f:
        return yaml.safe_load(f)

model_params = load_model_config("mtad_gat.yaml", EXP_ID)
dataset_params = load_dataset_config(DATASET_CFG_FILE)

print("=== Model params (subset) ===")
for k in ["dataset_id", "batch_size", "window_size", "stride", "nb_epoch"]:
    if k in model_params:
        print(f"{k}: {model_params[k]}")

print("\n=== Dataset params (subset) ===")
for k in ["dataset", "data_root", "dim", "entities"]:
    if k in dataset_params:
        v = dataset_params[k]
        if k == "entities" and isinstance(v, list):
            print(f"{k}: list(len={len(v)}) first5={v[:5]}")
        else:
            print(f"{k}: {v}")


=== Model params (subset) ===
dataset_id: SMD
batch_size: 512
window_size: 100
stride: 1
nb_epoch: 10

=== Dataset params (subset) ===



## 3) Load data and generate windows (mtad-tools)

This uses your existing MTAD-GAT utilities:
- `load_dataset` from `dataloader.py`
- `generate_windows` from `data_preprocess.py`

It creates:
- `train_windows`: shape `[N_train, window_size, dim]`
- `test_windows`:  shape `[N_test,  window_size, dim]`
- `test_labels`:   shape `[N_test,  window_size]` (point labels per time-step)

We then convert point labels to window labels for AUROC:
- window label = 1 if any point label in the window is 1, else 0


In [15]:
##new

In [16]:
# %%
import os
import re
from dataloader import load_dataset
from data_preprocess import generate_windows

DATA_ROOT = r"C:\Users\S4027872\OneDrive - RMIT University\Desktop\My-Study\Anomaly detection\code\mtad-gat-independent\SMD"
DIM = 38
VALID_RATIO = float(dataset_params.get("valid_ratio", 0.0))

window_size = int(model_params.get("window_size", 60))
stride = int(model_params.get("stride", 10))

print("DATA_ROOT:", DATA_ROOT)
print("Listing files in DATA_ROOT ...")

files = sorted([f for f in os.listdir(DATA_ROOT) if os.path.isfile(os.path.join(DATA_ROOT, f))])
print(f"Found {len(files)} files. First 30:")
for f in files[:30]:
    print("  ", f)

# ---------------------------
# Auto-detect common patterns
# ---------------------------
# We look for:
#   <entity>_train.<ext>
#   <entity>_test.<ext>
#   <entity>_test_label.<ext>
#
# ext could be: npy, pkl, pickle, or even no ext.
pattern = re.compile(r"^(?P<entity>.+)_(?P<split>train|test|test_label)\.(?P<ext>npy|pkl|pickle)$")

matches = [pattern.match(f) for f in files]
matches = [m for m in matches if m is not None]

if len(matches) == 0:
    # If no match, print a hint: maybe they are in subfolders or different naming
    raise FileNotFoundError(
        "Could not auto-detect SMD file pattern in DATA_ROOT.\n"
        "Expected files like: <entity>_train.npy or <entity>_train.pkl (and test/test_label).\n"
        "Check the printed file list above.\n"
        "If your data is in a subfolder (e.g., DATA_ROOT/SMD or DATA_ROOT/processed), "
        "set DATA_ROOT to that folder."
    )

entities = sorted(set(m.group("entity") for m in matches))
exts = sorted(set(m.group("ext") for m in matches))

print("\nAuto-detected entities:", len(entities), "first10:", entities[:10])
print("Auto-detected extensions:", exts)

# Choose the most common extension among matches
from collections import Counter
ext_counter = Counter(m.group("ext") for m in matches)
ext = ext_counter.most_common(1)[0][0]
print("Using extension:", ext)

# Set postfixes according to your dataloader.py convention: f"{entity}_{postfix}"
TRAIN_POSTFIX = f"train.{ext}"
TEST_POSTFIX = f"test.{ext}"
TEST_LABEL_POSTFIX = f"test_label.{ext}"

print("Resolved postfixes:", TRAIN_POSTFIX, TEST_POSTFIX, TEST_LABEL_POSTFIX)

# Quick existence check for first entity
probe = entities[0]
for postfix in [TRAIN_POSTFIX, TEST_POSTFIX, TEST_LABEL_POSTFIX]:
    p = os.path.join(DATA_ROOT, f"{probe}_{postfix}")
    print("Check:", p, "->", os.path.exists(p))

# ---------------------------
# Load dataset
# ---------------------------
data_dict = load_dataset(
    data_root=DATA_ROOT,
    entities=entities,
    valid_ratio=VALID_RATIO,
    dim=DIM,
    test_label_postfix=TEST_LABEL_POSTFIX,
    test_postfix=TEST_POSTFIX,
    train_postfix=TRAIN_POSTFIX,
)

# ---------------------------
# Generate windows
# ---------------------------
try:
    window_dict = generate_windows(
        data_dict,
        window_size=window_size,
        stride=stride,
        train_ratio=1.0,
    )
except TypeError:
    window_dict = generate_windows(
        data_dict,
        window_size=window_size,
        stride=stride,
    )

entities_loaded = list(window_dict.keys())
print("\nEntities loaded into windows:", len(entities_loaded), "first5:", entities_loaded[:5])

ENTITY_INDEX = 0  # change to select a different entity
entity = entities_loaded[ENTITY_INDEX]
print("Using entity:", entity)

w = window_dict[entity]
train_windows = w["train_windows"]
test_windows = w["test_windows"]
test_labels_point = w["test_label"]
test_labels_window = (test_labels_point.sum(axis=1) > 0).astype(int)

print("train_windows:", train_windows.shape)
print("test_windows :", test_windows.shape)
print("test_labels_window anomaly_ratio:", float(test_labels_window.mean()))


2026-02-18 14:12:35,751 - INFO - Loading data from C:\Users\S4027872\OneDrive - RMIT University\Desktop\My-Study\Anomaly detection\code\mtad-gat-independent\SMD


DATA_ROOT: C:\Users\S4027872\OneDrive - RMIT University\Desktop\My-Study\Anomaly detection\code\mtad-gat-independent\SMD
Listing files in DATA_ROOT ...
Found 85 files. First 30:
   .DS_Store
   machine-1-1_test.pkl
   machine-1-1_test_label.pkl
   machine-1-1_train.pkl
   machine-1-2_test.pkl
   machine-1-2_test_label.pkl
   machine-1-2_train.pkl
   machine-1-3_test.pkl
   machine-1-3_test_label.pkl
   machine-1-3_train.pkl
   machine-1-4_test.pkl
   machine-1-4_test_label.pkl
   machine-1-4_train.pkl
   machine-1-5_test.pkl
   machine-1-5_test_label.pkl
   machine-1-5_train.pkl
   machine-1-6_test.pkl
   machine-1-6_test_label.pkl
   machine-1-6_train.pkl
   machine-1-7_test.pkl
   machine-1-7_test_label.pkl
   machine-1-7_train.pkl
   machine-1-8_test.pkl
   machine-1-8_test_label.pkl
   machine-1-8_train.pkl
   machine-2-1_test.pkl
   machine-2-1_test_label.pkl
   machine-2-1_train.pkl
   machine-2-2_test.pkl
   machine-2-2_test_label.pkl

Auto-detected entities: 28 first10: ['machi

2026-02-18 14:12:36,747 - INFO - Loading 28 entities done.
2026-02-18 14:12:36,751 - INFO - Train/Valid/Test: 708405/0/708420 lines.
2026-02-18 14:12:36,804 - INFO - Generating sliding windows (size 100).
2026-02-18 14:12:37,180 - INFO - Windows for train #: (28379, 100, 38)
2026-02-18 14:12:37,500 - INFO - Windows for test #: (28379, 100, 38)
2026-02-18 14:12:37,770 - INFO - Windows for train #: (23594, 100, 38)
2026-02-18 14:12:38,042 - INFO - Windows for test #: (23594, 100, 38)
2026-02-18 14:12:38,287 - INFO - Windows for train #: (23602, 100, 38)
2026-02-18 14:12:38,717 - INFO - Windows for test #: (23602, 100, 38)
2026-02-18 14:12:38,935 - INFO - Windows for train #: (23606, 100, 38)
2026-02-18 14:12:39,188 - INFO - Windows for test #: (23606, 100, 38)
2026-02-18 14:12:39,569 - INFO - Windows for train #: (23605, 100, 38)
2026-02-18 14:12:39,858 - INFO - Windows for test #: (23605, 100, 38)
2026-02-18 14:12:40,088 - INFO - Windows for train #: (23588, 100, 38)
2026-02-18 14:12:40

MemoryError: Unable to allocate 342. MiB for an array with shape (23591, 100, 38) and data type float32

In [12]:
from typing import Optional
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class OnTheFlyWindowDataset(Dataset):
    """
    Returns windows without materializing all windows in RAM.

    Input:
      ts: np.ndarray [N, K] float32/float64
      labels: None or np.ndarray [N] with point labels (0/1)

    Output per item:
      x: torch.FloatTensor [K, T, 1]   (MTGFlow format)
      y: int window_label (0/1)        (for AUROC)
      idx: int window index
    """
    def __init__(self, ts: np.ndarray, window_size: int, stride: int, labels: Optional[np.ndarray] = None):
        assert ts.ndim == 2, "ts must be [N, K]"
        self.ts = ts.astype(np.float32, copy=False)
        self.labels = None if labels is None else labels.astype(np.int64, copy=False)
        self.T = int(window_size)
        self.S = int(stride)
        self.N = ts.shape[0]
        self.K = ts.shape[1]

        self.n_windows = 1 + (self.N - self.T) // self.S
        if self.n_windows <= 0:
            raise ValueError("Time series shorter than window_size.")

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx: int):
        start = idx * self.S
        end = start + self.T

        w = self.ts[start:end, :]  # [T, K]
        x = torch.from_numpy(w).transpose(0, 1).unsqueeze(-1)  # [K, T, 1]

        if self.labels is None:
            y = 0
        else:
            y = int(self.labels[start:end].sum() > 0)

        return x, y, idx

# --- create datasets ---
train_ds = OnTheFlyWindowDataset(train_ts, window_size=window_size, stride=stride, labels=None)
test_ds  = OnTheFlyWindowDataset(test_ts,  window_size=window_size, stride=stride, labels=test_label_point)

print("Train windows:", len(train_ds), "Test windows:", len(test_ds))

# --- create loaders ---
train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,  # safest on Windows
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


NameError: name 'train_ts' is not defined


## 4) Create PyTorch datasets and dataloaders for MTGFlow

MTGFlow expects input per batch as:
- `x`: `[B, K, T, 1]` where `K=dim` (number of sensors/features), `T=window_size`.

We treat each SMD entity (machine) as one multivariate system with `dim` sensors.


DEVICE: cpu


NameError: name 'train_loader' is not defined

In [10]:

# %%
from typing import Optional
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class OnTheFlyWindowDataset(Dataset):
    """
    Returns windows without materializing all windows in RAM.

    Input:
      ts: np.ndarray [N, K] float32/float64
      labels: None or np.ndarray [N] with point labels (0/1)

    Output per item:
      x: torch.FloatTensor [K, T, 1]   (MTGFlow format)
      y: int window_label (0/1)        (for AUROC)
      idx: int window index
    """
    def __init__(self, ts: np.ndarray, window_size: int, stride: int, labels: Optional[np.ndarray] = None):
        assert ts.ndim == 2, "ts must be [N, K]"
        self.ts = ts.astype(np.float32, copy=False)
        self.labels = None if labels is None else labels.astype(np.int64, copy=False)
        self.T = int(window_size)
        self.S = int(stride)
        self.N = ts.shape[0]
        self.K = ts.shape[1]

        self.n_windows = 1 + (self.N - self.T) // self.S
        if self.n_windows <= 0:
            raise ValueError("Time series shorter than window_size.")

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx: int):
        start = idx * self.S
        end = start + self.T

        w = self.ts[start:end, :]  # [T, K]
        x = torch.from_numpy(w).transpose(0, 1).unsqueeze(-1)  # [K, T, 1]

        if self.labels is None:
            y = 0
        else:
            y = int(self.labels[start:end].sum() > 0)

        return x, y, idx

# --- create datasets ---
train_ds = OnTheFlyWindowDataset(train_ts, window_size=window_size, stride=stride, labels=None)
test_ds  = OnTheFlyWindowDataset(test_ts,  window_size=window_size, stride=stride, labels=test_label_point)

print("Train windows:", len(train_ds), "Test windows:", len(test_ds))

# --- create loaders ---
train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,  # safest on Windows
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)



NameError: name 'train_ts' is not defined


## 5) Import MTGFlow model from the authors’ repo


In [6]:

# %%
# Add MTGFlow repo to path
if MTGFLOW_REPO_PATH not in sys.path:
    sys.path.insert(0, MTGFLOW_REPO_PATH)

from models.MTGFLOW import MTGFLOW

# Create MTGFlow model (using authors’ defaults)
# n_blocks: paper uses 1 for SWaT, 2 for others; here we expose it
N_BLOCKS = 2

model = MTGFLOW(
    n_blocks=N_BLOCKS,
    input_size=1,
    hidden_size=32,
    n_hidden=1,
    window_size=T,
    n_sensor=K,
    dropout=0.0,
    model="MAF",
    batch_norm=False,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("MTGFlow parameters:", f"{n_params:,}")


NameError: name 'T' is not defined


## 6) Train MTGFlow and evaluate AUROC

MTGFlow outputs log-likelihood.  
We use **negative log-likelihood (NLL)** as anomaly score (higher = more anomalous) for AUROC.


In [ ]:

# %%
from torch.nn.utils import clip_grad_value_

def evaluate_auc(model, loader):
    model.eval()
    scores = []
    labels = []
    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(DEVICE)
            # model.test returns log-likelihood per sample
            nll = (-model.test(x)).detach().cpu().numpy().reshape(-1)
            scores.append(nll)
            labels.append(np.asarray(y, dtype=np.int64).reshape(-1))
    scores = np.concatenate(scores)
    labels = np.concatenate(labels)
    auc = roc_auc_score(labels, scores)
    return auc, scores, labels

optimizer = torch.optim.Adam(
    [{"params": model.parameters(), "weight_decay": MTGFLOW_WEIGHT_DECAY}],
    lr=MTGFLOW_LR,
    weight_decay=0.0,
)

history = {"train_nll": [], "test_auc": []}
best_auc = -1.0
best_state = None

for epoch in range(MTGFLOW_EPOCHS):
    model.train()
    losses = []

    for x, _, _ in train_loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()
        loss = -model(x)  # maximize log-likelihood
        loss.backward()
        clip_grad_value_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        losses.append(float(loss.item()))

    train_nll = float(np.mean(losses))
    test_auc, test_scores, test_labels = evaluate_auc(model, test_loader)

    history["train_nll"].append(train_nll)
    history["test_auc"].append(float(test_auc))

    if test_auc > best_auc:
        best_auc = float(test_auc)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch+1:03d}/{MTGFLOW_EPOCHS} | train NLL={train_nll:.4f} | test AUROC={test_auc:.4f} | best={best_auc:.4f}")

# restore best state
if best_state is not None:
    model.load_state_dict(best_state)



## 7) Plot training curves


In [ ]:

# %%
plt.figure()
plt.plot(history["train_nll"])
plt.title("Train Negative Log-Likelihood")
plt.xlabel("Epoch")
plt.ylabel("NLL")
plt.show()

plt.figure()
plt.plot(history["test_auc"])
plt.title("Test AUROC")
plt.xlabel("Epoch")
plt.ylabel("AUROC")
plt.ylim(0.0, 1.0)
plt.show()

test_auc, test_scores, test_labels = evaluate_auc(model, test_loader)
print("Final (best-restored) Test AUROC:", test_auc)



## 8) Visualize anomaly score distribution


In [ ]:

# %%
plt.figure()
plt.hist(test_scores[test_labels == 0], bins=60, alpha=0.7, label="Normal")
plt.hist(test_scores[test_labels == 1], bins=60, alpha=0.7, label="Anomaly")
plt.title("MTGFlow anomaly scores on test windows (NLL)")
plt.xlabel("Anomaly score (higher = more anomalous)")
plt.ylabel("Count")
plt.legend()
plt.show()



## 9) Inspect top anomalous windows (optional)

This helps you debug and compare with MTAD-GAT results.


In [ ]:

# %%
top_k = 5
top_idx = np.argsort(-test_scores)[:top_k]
print("Top anomalous window indices:", top_idx.tolist())
print("Scores:", [float(test_scores[i]) for i in top_idx])
print("Labels:", [int(test_labels[i]) for i in top_idx])

# Plot a few sensors for each top window
def get_window(ds: MTGFlowWindowDataset, i: int):
    x, y, _ = ds[i]
    return x.numpy(), y  # x: [K,T,1]

for rank, i in enumerate(top_idx, start=1):
    xw, y = get_window(test_ds, int(i))
    K, T, _ = xw.shape
    n_plot = min(5, K)

    plt.figure()
    for k in range(n_plot):
        plt.plot(np.arange(T), xw[k, :, 0], label=f"sensor {k}")
    plt.title(f"Top-{rank} anomalous window (idx={i}, label={y}, score={test_scores[i]:.3f})")
    plt.xlabel("t in window")
    plt.ylabel("value (normalized by mtad-tools preprocessing)")
    plt.legend()
    plt.show()
